In [1]:
class GPT:
    def __init__(self,num_dims,num_heads,num_layers,vocab_size,seq_len):
        self.seq_len = seq_len
        self.wte = TokenEmbeddings(vocab_size,num_dims)
        self.wpe = PositionalEmbeddings(seq_len,num_dims)

        self.blocks = [Transformer(num_dims,num_heads) for _ in range(num_layers)]
        
        self.ln_f = LayerNorm(num_dims)
        self.lm_head = LinearLayer(num_dims,vocab_size)

    def forward(self,idx):
        B,T = idx.shape

        pos = torch.arange(T, device=idx.device)

        tok_emb = self.wte.forward(idx)
        pos_emb = self.wpe.forward(pos)

        x = tok_emb+pos_emb

        for block in self.blocks:
            x = block.forward(x)

        x  = self.ln_f.forward(x)
        logits = self.lm_head.forward(x)

        return logits


    def backward(self, grad_out):
      
        grad_logits = self.lm_head.backward(grad_out)
        grad_ln = self.ln_f.backward(grad_logits)


        grad = grad_ln
        for block in reversed(self.blocks):
            grad = block.backward(grad)

        self.wpe.backward(grad)
        self.wte.backward(grad)

        return grad
        
    

### GPT Block — Pre-LN Architecture

Every module up to this point was an individual paid actor 😂️. Now we will bring all of them into a single show!

As you can see below, this is the order of flow. We will instantiate the classes we have written above and after that we can start the flow.

### 1. What the Whole Assembly Line Looks Like

```
 ├─> wte (Token Embeddings) ─────┐
 │ ▼
 └─> wpe (Position Embeddings) ─> (+) = x : (B, T, d_model)
 │
 ▼
 ┌─────────────────────┐
 │ TransformerBlock 1 │
 └──────────┬──────────┘
 ▼
 ┌─────────────────────┐
 │ TransformerBlock 2 │
 └──────────┬──────────┘
 ▼
 [ ... ]
 ▼
 ┌─────────────────────┐
 │ TransformerBlock N │
 └──────────┬──────────┘
 │
 ▼
 Final LN (ln_f)
 │
 ▼
 LM Head (Linear)
 │
 ▼
 Logits: (B, T, vocab_size)
```

### 2. Initializing the GPT Model

We are in the GPT class. We instantiate objects for the Token Embeddings (`wte`) and Positional Embeddings (`wpe`).

After that, we create the stack of Transformer blocks using a list comprehension.

Finally, we create the two final objects:
1. **Final LayerNorm (`ln_f`)**: Normalizes final representations across feature dimensions before predicting tokens.
2. **Ultimate LM Head (`lm_head`)**: A linear layer that projects token representations from `d_model` dimensions into `vocab_size` logits to generate tokens.

### 3. Forward Pass Flow

We pass token indices into the `forward` method, extract `(B, T)` shapes, generate positional indices `[0, 1, ..., T-1]`, sum token and position embeddings, pass through all Transformer blocks, apply final LayerNorm and compute vocabulary logits.
